In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
df = pd.read_csv(os.path.join(path,"Q3_data.csv"))

In [ ]:
# Task 2: Write your code here:
display(df.head())

In [ ]:
# Task 3: Write your code here:
display(df.info())

In [ ]:
# Task 4: Write your code here:
display(df.describe())

In [ ]:
# Task 1: Write your code here:
df_clean = df.copy()

for col in df.columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

display(df_clean)
print(df_clean[col].isnull().sum())

In [ ]:
# Task 2: Write your code here:
duplicates = df_clean.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
display(df_clean)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
print(f"Categorical Columns: {categorical_cols}")

In [ ]:
# Task 4: Write your code here:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
X = df_clean.drop(['Target'],axis=1)
y= df_clean["Target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
#NOTE: I used train test split here before it is asked because I want to emphasize that the scaling for the test set should not be fitted due to the data leakage as the TA's taught us

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns
print(df["Target"].value_counts(normalize=True))
plt.figure(figsize=(8, 5))
sns.countplot(x=df["Target"])
plt.title("Target Distribution")
plt.show()

print("Clearly imbalanced")

In [ ]:
# Task 1: Write your code here:
# Already done above :)

In [ ]:
# Task 2,3,4,5: Write your code here:
all_results = {'accuracy': [], 'f1': []}

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y  # IMPORTANT: Preserves class distribution
)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)
  # 2. Train & Validate sklearn models

  print(f"Training {model}...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_results['accuracy'].append(accuracy)
  all_results['f1'].append(f1)

  print(f"  Accuracy:  {np.mean(all_results['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
importances = {}

importances['CatBoost'] = model.feature_importances_

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
sorted_idx = np.argsort(importances['CatBoost'])
features = X.columns
print(features[sorted_idx], importances["CatBoost"][sorted_idx])

In [ ]:
# Task Bonus: Write your code here: